# Programmieraufgabe 6: cg-Verfahren und Vorkonditionierung

**Abgabe in den Programmiertutorien am 30./31. Juli 2025.**

Benötigte Module:

In [ ]:
import numpy as np
import scipy.sparse as spsp      # zum Speichern von und Rechnen mit dünn-besetzten Matrizen
import matplotlib.pyplot as plt  # zum Erstellen von Plots
import numpy.random as rnd       # für alles, was mit Zufallszahlen zu tun hat

## Modellmatrix und Hintergründe

Für $N \in \mathbb{N}$ mit $N\gg1$ betrachten wir die $N^2\times N^2$-Matrix
$$ A = \begin{pmatrix} 
     D_N & -I_N \\
    -I_N &  D_N & -I_N \\
         &\ddots&\ddots&\ddots \\
         &      & -I_N &  D_N & - I_N \\
         &      &      & -I_N &  D_N
\end{pmatrix} 
\qquad \text{mit} \qquad
D_N = \begin{pmatrix}
     4 & -1   \\
    -1 &  4   & -1   \\
       &\ddots&\ddots&\ddots \\
       &      & -1   &  4   & -1 \\
       &      &      & -1   &  4
\end{pmatrix} \in \mathbb{R}^{N\times N}
$$
und mit der Einheitsmatrix $I_n \in \mathbb{R}^{N\times N}$. 

Solche Matrizen treten bei der Raumdiskretisierung partieller Differentialgleichungen auf und sind damit tatsächlich praxisrelevant. Details dazu können Sie in den Vorlesungen _Numerische Methoden für Differentialgleichungen_ und/oder _Einführung in das Wissenschaftliche Rechnen_ erlernen.

Die Matrix $A$ ist dünn-besetzt, d.h. sie hat im Verhältnis zu ihrer Größe nur sehr wenige Nicht-Null-Einträge (konkret sind nur ca. $5N^2$ von $N^4$ Einträgen Nicht-Null). Leider gilt das nicht für die Matrix-Zerlegungen, die Sie in Numerik 1 kennengelernt haben (LU, Cholesky, QR). Für große Werte von $N$ sind direkte Löser zur Berechnung der Lösung linearer Gleichungssysteme mit der Matrix $A$ daher zu aufwendig. Als Alternative wurden in der Vorlesung Krylov-Verfahren zur Approximation von Lösungen diskutiert. Diese basieren nur auf Matrix-Vektor-Produkten mit der Matrix $A$. Solche Produkte sind dank der dünnen Besetzung der Matrix $A$ sehr günstig.

Die Matrix $A$ ist offensichtlich symmetrisch. Außerdem ist sie auch positiv definit (dass alle Eigenwerte $\geq0$ sind folgt z.B. direkt aus dem Satz von Gershgorin, was schonmal die positive Semidefinitheit begründet). Damit eignet sich insbesondere das cg-Verfahren.

Die folgende Prozedur erstellt die $\mathbb{R}^{N^2 \times N^2}$-Matrix $A$ für vorgegebenes $N$. Sie brauchen die Prozedur nicht genauer zu verstehen. Die Matrix wird in einem speziellen Speicherformat für dünn-besetzte Matrizen gespeichert. Das spart einerseits sehr viel Speicherplatz (es werden nämlich tatsächlich nur die ca. $5N^2$ Einträge gespeichert, statt alle $N^4$ Einträge), und andererseits erlaubt es die effiziente Berechnung von Matrix-Vektor-Produkten mit der Matrix $A$. 

In [ ]:
def delsq(N):
    A = spsp.diags_array([-1, 2, -1], offsets=[-1, 0, 1], shape=(N, N))
    A = spsp.kron(A,spsp.eye_array(N)) + spsp.kron(spsp.eye_array(N),A)
    return A.tocsc()

Für die Umsetzung der von Ihnen zu bearbeitenden Aufgabenteile spielt das Speicherformat keine Rolle. Sie können mit der Matrix $A$ umgehen wie Sie das bisher auch immer getan haben. Insbesondere können Matrix-Vektor-Produkte wie üblich über `A @ ...` berechnet werden.

Für Interessierte geben wir die Matrix $A$ für $N=3$ (also eine $9\times9$-Matrix) spaßeshalber einmal aus:

In [ ]:
A = delsq(3)
print(A)

Die erste Zeile `(0, 0)	   4.0` bedeutet zum Beispiel, dass an der Stelle `(0,0)` der Matrix (also ganz links oben) ein Nicht-Null-Eintrag steht, und zwar der Eintrag `4.0`. Analog in den weiteren Zeilen. Es werden also letztendlich zwei Vektoren mit Zeilen- und Spaltenindizes gespeichert, sowie ein Vektor mit den Werten der Einträge. Das Format ist zwar schwer zu lesen, aber dafür eben speichereffizient.

Mit der Methode `toarray()` wird die Matrix zu einer vollbesetzten Matrix umgewandelt:

In [ ]:
print(A.toarray())

## Referenzlösung
Mit dem cg-Verfahren soll später die Lösung $\widehat{x}$ des linearen Gleichungssystems $Ax=b$ für eine gegebene rechte Seite $b\in\mathbb{R}^{N^2}$ approximiert werden. Damit wir das Ergebnis des Verfahrens überprüfen können, geben wir uns die Lösung $\widehat{x}$ selbst vor (und zwar als zufälligen Vektor), und berechnen die dazu passende rechte Seite durch $b = A\widehat{x}$:

In [ ]:
N = 50
A = delsq(N)
xhat = rnd.rand(N**2)
b = A@xhat

## cg-Verfahren 

**(a) Implementieren Sie das cg-Verfahren zur Approximation der Lösung eines LGS $Ax=b$ mit einer symmetrischen, positiv definiten Matrix $A$.**

Beachten Sie dabei folgendes:
- Es reicht, wenn Ihre Prozedur mit reellwertigen Matrizen umgehen kann.
- Neben der Matrix $A$ und dem Vektor $b$ soll die Prozedur folgende Eingabeparameter erhalten:
    - eine Anfangsnäherung `x0`,
    - eine Toleranz `tol`,
    - eine maximale Anzahl an Iterationen `maxIt`.
- Zurückgegeben werden sollen:
    - die finale Approximation an die Lösung,
    - die Anzahl durchgeführter Iterationen,
    - ein Vektor, der die Norm des Residuums von allen Iterierten enthält.
- Falls das Verfahren vor Erreichen der Toleranz abbricht, weil die maximale Anzahl an Iterationen erreicht wurde, sollen die selben Größen zurückgegeben werden. Zusätzlich soll eine geeignete Meldung auf dem Bildschirm ausgegeben werden.

**(b) Testen Sie Ihre Prozedur, indem Sie sie mit der oben erstellten Matrix $A$ und rechten Seite $b$ aufrufen.**

Starten Sie mit dem Nullvektor, wählen Sie $10^{-6}$ als Toleranz für das Residuum, und setzen Sie die maximale Anzahl an Iterationen auf $200$. Geben Sie die finale Norm des Residuums, die Anzahl berechneter Approximationen, sowie den Fehler der finalen Approximation gegenüber der Referenzlösung $\widehat{x}$ aus. 

Die Toleranz sollte nach weniger als $200$ Iterationen erreicht werden, und die Norm des Residuum am Ende dementsprechend tatsächlich kleiner als $10^{-6}$ sein.

**(c) Plotten Sie die Norm des Residuums in Abhängigkeit der Anzahl durchgeführter Iterationen. Tragen Sie dazu das Residuum auf einer logarithmischen Achse ab (dazu ist der Befehl `plt.semilogy(...)` da).**

## cg-Verfahren mit Vorkonditionierung

Als nächstes wollen wir das cg-Verfahren um einen Vorkonditionierer erweitern, um die Konvergenz zu beschleunigen. Konkret nutzen wir den Vorkonditionierer $B = CC^T$ mit $C = \widetilde{L}^{-T}$, wobei $\widetilde{L}$ die linke untere Dreiecksmatrix aus der unvollständigen Cholesky-Zerlegung von $A$ ist. Die Matrix $\widetilde{L}$ und deren Transponierte werden mit der folgenden Prozedur berechnet (auch diese müssen Sie nicht nachvollziehen, sie ist aber sehr nah dran am Algorithmus, den Sie aus der Vorlesung kennen):

In [ ]:
def ichol(A):
    N = np.size(A,1)
    L = spsp.tril(A,format='csr')
    # L = spsp.csr_matrix((N, N))
    for j in range(N): # Spalte für Spalte von L berechnen  
        # Berechnung von L[j,j]:
        L[j,j] = A[j,j]
        inds1 = L[[j],:j].indices # Indizes der Spalten von L, in denen Nicht-Nulleinträge in der j-ten Zeile von L stehen
        for k in inds1:
            L[j,j] -= L[j,k]**2
        L[j,j] = np.sqrt(L[j,j])
        
        # Berechnung von L[i,j] für alle i>j:
        inds2 = A[:,[j]].indices # Indizes der Zeilen von A, in denen Nicht-Nulleinträge in der j-ten Spalte stehen
        inds2 = inds2[inds2>j]
        for i in inds2:
            L[i,j] = A[i,j]
            for k in inds1:
                L[i,j] -= L[i,k]*L[j,k]
            L[i,j] /= L[j,j]

        LT = np.transpose(L).tocsr()
    return L,LT

Die Berechnung der unvollständigen Cholesky-Zerlegung kann einige Sekunden dauern:

In [ ]:
L,LT = ichol(A)

Anwendung des Vorkonditionieres bedeutet letztendlich einfach, dass wir das cg-Verfahren auf die Matrix $\widetilde{A} = C^T A C$ und den Vektor $\widetilde{b} = C^Tb$ anwenden. Im cg-Verfahren sind dann Matrix-Vektor-Multiplikationen mit der Matrix $\widetilde{A}$ nötig. Die Matrix $\widetilde{A}$ wird dafür aber nicht explizit ausgerechnet, sondern es wird erst mit der Matrix $C$, dann mit $A$, dann mit $C^T$ multipliziert. Da $C=\widetilde{L}^{-T}$ gilt, muss für die Multiplikation mit $C$ bzw. $C^T$ ein LGS mit einer dünn-besetzten Dreiecksmatrix gelöst werden. Dazu gibt es im `sparse`-Modul die Prozedur `linalg.spsolve_triangular`.

**(d) Schreiben Sie eine Prozedur `mult_Atilde(A,L,LT,u)`, welche das Matrix-Vektor-Produkt $C^TACu$ mit $C = \widetilde{L}^{-T}$ mithilfe der Prozedur `spsolve_triangular` (Aufruf über `spsp.linalg.spsolve_triangular(...)`) berechnet und zurückgibt.**

Beachten Sie, dass Sie bei der Prozedur `spsolve_triangular` einstellen müssen, ob es sich um ein LGS mit einer oberen oder unteren Dreiecksmatrix handelt.

**(e) Kopieren Sie Ihre Prozedur `cg` von oben und ändern Sie sie zu einer Prozedur `pcg` (preconditioned cg) ab, in der das cg-Verfahren mit der unvollständigen Cholesky-Zerlegung als Vorkonditionierer angewandt wird. Die Marizen $\widetilde{L}$ und $\widetilde{L}^T$ sollen der Prozedur dabei als zusätzliche Eingabeparameter übergeben werden.**

Nutzen Sie dabei die Prozedur `mult_Atilde` aus Teil (d). Beachten Sie, dass auch die rechte Seite $b$ vor Start der Iterationen angepasst werden muss.

**(f) Testen Sie Ihre Prozedur wie in Teil (b).** 

Beachten Sie, dass mit dem vorkonditionierten cg-Verfahren die Lösung $\widetilde{x}$ des LGS $\widetilde{A} x = \widetilde{b}$ approximiert wird (und nicht des LGS $Ax=b$). Erinnern Sie sich an die Vorlesung, wie die Lösung beider LGS zusammenhängen, bevor Sie den Fehler gegenüber der Referenzlösung $\widehat{x}$ berechnen.

**(g) Erstellen Sie einen Plot wie in Teil (c), bei dem die Entwicklung der Norm der Residuen in Abhängigkeit der Anzahl an Iterationen sowohl für das cg- als auch das vorkonditionierte cg-Verfahren dargestellt sind. Was beobachten Sie?** 

Zum Abschluss noch eine **Warnung**: 

Der von Ihnen eben erzeugte Plot suggeriert (hoffentlich), dass das vorkonditionierte cg-Verfahren mit weniger Iterationen kleine Residuen erreicht als das cg-Verfahren ohne Vorkonditionierung. Andererseits ist jede Iteration des vorkonditionierten cg-Verfahrens teurer, denn man muss ja zusätzlich zwei LGS mit dünn-besetzten Dreiecksmatrizen lösen. Eigentlich müsste man fairerweise die Norm des Residuums über die Laufzeit plotten, und dann würde das vorkonditionierte cg-Verfahren vielleicht keinen Vorteil mehr bringen! Um einen solchen Plot zu erstellen, müsste man beide Verfahren möglichst effizient implementieren, und das ist gerade beim Umgang mit dünn-besetzten Matrizen alles Andere als einfach. 

Letztendlich sollten Sie folgende Erkenntnis mitnehmen: Vorkonditionierung _kann das Potential haben,_ eine (manchmal auch enorme) Konvergenzbeschleunigung liefern, das hängt aber von der Matrix, dem gewählten Vorkonditionierer, den gewünschten Toleranzen, usw. ab.